# 🎵 Estudio de Música Digital — Separar instrumentos con IA

Este notebook separa una canción en pistas individuales (voz, batería, bajo, piano, guitarra, otros) usando **Demucs** (Meta AI), el mejor modelo abierto para esto hoy.

**Por qué en Colab y no en el teléfono/servidor de VAWOL:** este modelo necesita GPU para andar en minutos en vez de correr el riesgo de colgarse por falta de RAM. Colab te da una GPU gratis (sin tarjeta de crédito) — este es el camino de costo cero.

**Cómo usarlo:**
1. Arriba a la derecha: `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → elegí **GPU (T4)**.
2. Corré las celdas de arriba a abajo (▶️ en cada una, o `Entorno de ejecución` → `Ejecutar todas`).
3. Cuando pida el archivo, subí tu canción (mp3, wav, etc.).
4. Al final se descarga un .zip con cada instrumento en su propio archivo de audio.

Con esos archivos separados ya podés llevarlos al Estudio de Música Digital (o a cualquier editor) y trabajar cada instrumento por separado — por ejemplo, quedarte solo con el piano.

In [ ]:
# Verificar que hay GPU (si dice "no GPU", volvé al paso 1 de arriba)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "⚠️ SIN GPU: anda a Entorno de ejecución > Cambiar tipo > GPU"

In [ ]:
# Instalar Demucs (tarda ~1-2 minutos)
!pip install -q demucs

In [ ]:
# Subir la canción
from google.colab import files
import os

subidos = files.upload()
nombre_archivo = list(subidos.keys())[0]
print(f"\n✓ Subido: {nombre_archivo}")

## Elegí el modelo

- **`htdemucs_6s`** (recomendado si querés el **piano** o la **guitarra** separados): 6 pistas — voz, batería, bajo, guitarra, piano, otros. Es un modelo experimental; a veces se "filtra" un poco de otro instrumento en la pista de piano/guitarra.
- **`htdemucs_ft`**: el de mejor calidad general, pero solo 4 pistas — voz, batería, bajo, otros (el piano queda mezclado dentro de "otros"). Tarda un poco más (corre 4 modelos y promedia).

In [ ]:
MODELO = "htdemucs_6s"  # cambiar a "htdemucs_ft" si no necesitás piano/guitarra separados

!demucs -n {MODELO} "{nombre_archivo}"

In [ ]:
# Empaquetar los resultados y descargar
import shutil

base = os.path.splitext(nombre_archivo)[0]
carpeta = f"separated/{MODELO}/{base}"
print("Pistas generadas:", os.listdir(carpeta))

zip_path = f"{base}_instrumentos"
shutil.make_archive(zip_path, "zip", carpeta)
files.download(f"{zip_path}.zip")

---
### Notas honestas
- **Tiempo**: con GPU T4, una canción de 3-4 minutos tarda típicamente 1-3 minutos en separarse (vs. varios minutos por CPU, sin GPU).
- **Cuota gratis de Colab**: variable según uso de Google, pero suele rondar 12-30 horas de GPU por semana sin pagar nada. Para separar canciones sueltas alcanza de sobra.
- **Calidad**: Demucs es el estado del arte abierto hoy, pero ningún separador es perfecto — vas a escuchar algo de "bleed" (residuos de otros instrumentos), sobre todo en el modelo de 6 pistas.
- Si la sesión de Colab se desconecta por inactividad, simplemente volvé a correr las celdas desde el principio (los archivos subidos se pierden al reiniciar).